In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import json

%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from utils import plot_figures

# Plot from data files
#### Run multiple seeds

### Parameters

In [ ]:


import os
import json
import re

data_dir = "data_20_seeds_no_Leap"
all_seeds_histories = {}

pattern = re.compile(r"history_seed_(\d+)\.json")
# print(pattern)

for filename in os.listdir(data_dir):
    match = pattern.match(filename)
    if match:
        print("Reading ", filename)
        seed = int(match.group(1))
        
        filepath = os.path.join(data_dir, filename)
        with open(filepath, "r") as f:
            all_seeds_histories[seed] = json.load(f)



In [ ]:
all_seeds = list(all_seeds_histories.keys())
print(len(all_seeds ))


In [ ]:
# print(histories.keys())
# print(histories["Gradient Method"].keys())
# print(histories["GLAd-SSN, p=0.2"].keys())

template_seed = all_seeds[0]

averaged_history = {
    method_name: {
        measure: np.zeros_like(
            np.array(all_seeds_histories[template_seed][method_name][measure]), dtype=np.float64
        )
        for measure in all_seeds_histories[template_seed][method_name]
    }
    for method_name in all_seeds_histories[template_seed]
}

print(averaged_history.keys())
 

In [ ]:
def add_common(a, b):
    n = min(len(a), len(b))
    return a[:n] + b[:n]


for seed in all_seeds:
    for method_name in all_seeds_histories[seed].keys():
        for measure in all_seeds_histories[seed][method_name].keys():
            # print(method_name)   
            # print(measure)
            # print(len(all_seeds_histories[10][method_name][measure]))
            # print(len(all_seeds_histories[20][method_name][measure]))
            averaged_history[method_name][measure] =  add_common(np.array(averaged_history[method_name][measure]), np.array(all_seeds_histories[seed][method_name][measure]))
# Divide
for method_name in averaged_history:
    for measure in averaged_history[method_name]:
        averaged_history[method_name][measure] /= len(all_seeds)
        averaged_history[method_name][measure] = averaged_history[method_name][measure].tolist()  # Convert to list for JSON serialization

filename="data/averaged_history"
for seed in all_seeds:
    filename = filename + "_" + str(seed)
filename = filename + ".json"

print("Saving averaged data as ", filename)
with open(filename, 'w') as f:
    json.dump(averaged_history, f)

# Code for plotting

In [ ]:
from utils import plot_figures
methods_to_run = list(averaged_history.keys())
print(methods_to_run)
histories = averaged_history
resid = np.array(list(histories.values())[1]["func"])
resid_2 = np.array(list(histories.values())[1]["func_at_lssolve"])


line_widths = [2.5] * len(methods_to_run)
line_styles = ["-"] * len(methods_to_run)

gradient_method_color = "#E8601C"
cubic_newton_color = "gold"
leapssn_color = "black"
gladssn_color = "#9B8AC4"
gladssn_1_color = "#88A5DD"
gladssn_2_color = "#8DCBE4"
gladssn_3_color = "#C2E3D2"
gladssn_4_color = "#DDECBF"
# gladssn_5_color = "#F2E1D0"
colors = [gradient_method_color, gladssn_color, gladssn_1_color, gladssn_2_color, gladssn_3_color, gladssn_4_color]
marker_cycle = [
"None",   # None Gradient Method
# "*",   # star Cubic Newton
# "s",   # square LeapSSN
"^",   # triangle up GladSSN
"D",   # diamond GladSSN second
"x",   # x GladSSN third
"v",   # triangle down GladSSN fourth
"h",   # hexagon GladSSN fifth
] 


list_of_mins = [np.min(history["func"]) for history in histories.values()]
mn_func = min(list_of_mins)
plot_figures(
    list(histories.values()),
    list(histories.keys()),
    colors=colors,
    linewidths=line_widths,
    linestyles=line_styles,
    n=100,
    m=None,
    mu_str=("0.5"),
    f_star=mn_func,  # mn_func,
    threshold=1e-10,
    suptitle=("Matrix Factorization"),
    which_measure_2="func",
    max_iter=1000,
)
plot_figures(
    list(histories.values()),
    list(histories.keys()),
    colors=colors,
    linewidths=line_widths,
    linestyles=line_styles,
    n=100,
    m=None,
    mu_str=("0.5"),
    f_star=0,
    threshold=1e-10,
    suptitle=("Matrix Factorization"),
    max_iter=1000,
    which_measure_1="grad_norm",
    which_measure_2="grad_norm",
    # which_measure_2="grad_norm_at_lssolve",
    ylabel="Gradient norm",
)

plt.show()

# Plots for the paper

In [ ]:
from utils import plot_for_paper_2, plot_for_paper_penalty
markevery_numbers = [50000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]

plot_for_paper_2(
    list(histories.values()),
    list(histories.keys()),
    colors=colors,
    linewidths=line_widths,
    linestyles=line_styles,
    n=100,
    m=None,
    mu_str=("0.5"),
    f_star=mn_func,
    threshold=1e-10,
    suptitle=("Plots"),
    max_iter=12000,
    which_measure="func",
    marker_cycle=marker_cycle,
    markevery_numbers=markevery_numbers,
)
plot_for_paper_2(
    list(histories.values()),
    list(histories.keys()),
    colors=colors,
    linewidths=line_widths,
    linestyles=line_styles,
    n=100,
    m=None,
    mu_str=("0.5"),
    f_star=mn_func,
    threshold=1e-10,
    suptitle=("Plots"),
    max_iter=12000,
    which_measure="grad_norm",
    marker_cycle=marker_cycle,
    markevery_numbers=markevery_numbers,
)
plot_for_paper_penalty(
    list(histories.values()),
    list(histories.keys()),
    colors=colors,
    linewidths=line_widths,
    linestyles=line_styles,
    marker_cycle=marker_cycle,
    max_time=12000,
)

In [ ]:
# Plot each individually

for seed in all_seeds:
    print("Seed number", seed)
    plot_for_paper_2(
        list(all_seeds_histories[seed].values()),
        list(all_seeds_histories[seed].keys()),
        colors=colors,
        linewidths=line_widths,
        linestyles=line_styles,
        n=100,
        m=None,
        mu_str=("0.5"),
        f_star=mn_func,
        threshold=1e-10,
        suptitle=("Plots for Seed " + str(seed)),
        max_iter=12000,
        which_measure="grad_norm",
        marker_cycle=marker_cycle,
        markevery_numbers=markevery_numbers,
    )
    plot_for_paper_penalty(
        list(all_seeds_histories[seed].values()),
        list(all_seeds_histories[seed].keys()),
        colors=colors,
        linewidths=line_widths,
        linestyles=line_styles,
        marker_cycle=marker_cycle,
        max_time=12000,
    )